# 04 · Compare the same questions across models

Use one ticket to compare **Jev 1.13, Jev 1.14, OpenAI Decisions, and OpenAI Responses (an LLM)**.
Keep the evidence, categories, and rubrics the same. Read the full probabilities, even when the selected answer stays the same.

Jev and Decisions return native probabilities. Our LLM request returns a category, a score level, and a Boolean—not a probability distribution. Its probability column is therefore unavailable, never zero or an invented confidence.

Run cells in order. Text comparison makes four requests; the optional image comparison makes four more, before retries. Missing keys skip the corresponding provider. Recorded mode requires matching authentic recordings.


In [ ]:
# Find the course folder and load API settings without printing secret keys.
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'course_utils').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open Jupyter inside the course folder.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from course_utils.config import load_settings, require_key
settings = load_settings(ROOT)
print('Python:', sys.version.split()[0], '| Mode:', settings.mode)
print('Jev key configured:', bool(settings.typesafe_api_key))
print('OpenAI key configured:', bool(settings.openai_api_key))

## 1. Define the questions
Choice selects a team; Score rates frustration; Noul checks for an explicit refund request.

In [ ]:
# Load the example tickets and define the team, frustration, and refund questions.
tickets = [json.loads(line) for line in (ROOT / 'data/tickets.jsonl').read_text().splitlines()]
TEAMS = {
    'billing': 'Existing payments, refunds, invoices, and subscriptions.',
    'technical': 'Malfunctioning application features or service errors.',
    'account': 'Login, credentials, profiles, and membership.',
    'other': 'Sales, suggestions, general questions, or insufficient evidence.',
}
FRUSTRATION = ['Neutral or positive.', 'Mild complaint or annoyance.',
               'Clear or repeated frustration.', 'Intense anger or explicit outrage.']
TEAM_PROMPT = 'Which team should handle the ticket? For mixed intents choose the first actionable issue.'
REFUND_PROMPT = 'Does the customer explicitly request money back? Policy questions, cancellation alone, and negated requests are not requests.'
from typesafe_sdk import TypeSafeClient, Choice, Score, Noul, RetryPolicy
questions = {
    'team': Choice(instructions=TEAM_PROMPT, criteria=TEAMS),
    'frustration': Score(instructions='How frustrated is the customer, based on expressed tone rather than urgency?', criteria=FRUSTRATION),
    'refund_requested': Noul(instructions=REFUND_PROMPT),
}

## 2. Set up the Jev request
The helper below sends the same questions for either model version and keeps an authentic record for replay.

In [ ]:
# Define helpers for calling Jev, saving results, and replaying recorded answers.
from course_utils.records import run_case
from contextlib import nullcontext
import hashlib
RUN_DIR = ROOT / 'work/runs'  # Change to recordings/runs for a supplied authentic recording set.
can_jev = settings.mode == 'recorded' or bool(settings.typesafe_api_key)
can_openai = settings.mode == 'recorded' or bool(settings.openai_api_key)
if not can_jev:
    print('Jev calls skipped: configure TYPESAFE_API_KEY. No model results will be invented.')

def evaluate_ticket(client, case, model, *, trial=0, warmup=False):
    state = {'ticket': case['text']}
    config = {'state': state, 'questions': {k:v.model_dump(mode='json') for k,v in questions.items()},
              'max_retries':1, 'timeout':30}
    return run_case(lambda: client.system_one(model=model, state=state, questions=questions),
        provider='typesafe', endpoint='systemone', model=model, example_id=case['id'],
        config=config, question_names=list(questions), mode=settings.mode, directory=RUN_DIR,
        trial=trial, warmup=warmup)

def jev_client():
    return (TypeSafeClient(api_key=require_key(settings,'typesafe'), retry=RetryPolicy(max_retries=1), timeout=30)
            if settings.mode=='live' else nullcontext(None))

## 3. Set up matching OpenAI requests
Decisions uses typed questions. Responses uses a JSON schema with the same categories and score rubric. A valid JSON answer can still be incorrect.

In [ ]:
# Import the OpenAI client and schema types.
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
import base64

In [ ]:
# Define the structured LLM answer for a ticket.
class TicketDecision(BaseModel):
    team: Literal['billing','technical','account','other']
    frustration: int = Field(ge=0,le=3)
    refund_requested: bool

In [ ]:
# Define the structured LLM answer for a screenshot.
class ImageDecision(BaseModel):
    category: Literal['payment','login','service','none']
    severity: int = Field(ge=0,le=3)
    error_present: bool

In [ ]:
# Translate the Jev questions into matching OpenAI Decisions questions.
def to_openai_questions(query):
    converted=[]
    for name,q in query.items():
        item={'name':name,'instructions':q.instructions}
        if q.type=='choice':
            item.update(type='choice',choices=[{'value':k,'description':v} for k,v in q.criteria.items()])
        elif q.type=='score':
            item.update(type='score',levels=[{'label':str(i),'description':text} for i,text in enumerate(q.criteria)])
        elif q.type=='noul':
            item.update(type='predicate')
        else:
            raise ValueError('This comparison supports Choice, Score, Noul; Box is separate.')
        converted.append(item)
    return converted

In [ ]:
# Prepare the evidence once, so both OpenAI endpoints receive the same input.
def prepare_openai_input(case, kind):
    if kind == 'text':
        evidence = json.dumps({'ticket': case['text']})
        identity = {'input': evidence}
        return evidence, identity, questions, TicketDecision

    path = ROOT / 'data' / case['image_path']
    image_bytes = path.read_bytes()
    encoded = base64.b64encode(image_bytes).decode('ascii')
    evidence = [{'role': 'user', 'content': [
        {'type': 'input_text', 'text': 'Evaluate the screenshot.'},
        {'type': 'input_image', 'image_url': 'data:image/png;base64,' + encoded, 'detail': 'high'},
    ]}]
    identity = {
        'image_sha256': hashlib.sha256(image_bytes).hexdigest(),
        'input_text': 'Evaluate the screenshot.',
        'detail': 'high',
    }
    return evidence, identity, IMAGE_QUESTIONS, ImageDecision

In [ ]:
# Build a Decisions request using typed questions and native probabilities.
def decisions_request(client, evidence, typed_questions):
    model = settings.openai_decisions_model
    config = {'questions': typed_questions}

    def request():
        return client.decisions.create(
            model=model,
            input=evidence,
            questions=typed_questions,
        )

    return request, model, config

In [ ]:
# Build an LLM request for a strict JSON answer, without asking it to invent probabilities.
def llm_request(client, evidence, typed_questions, schema):
    model = settings.openai_model
    instructions = (
        'Evaluate only the supplied evidence. Return the fields required by the schema. '
        'Use the following questions and rubrics; for score fields select one level index. '
        + json.dumps(typed_questions)
    )
    output_format = {
        'type': 'json_schema',
        'name': schema.__name__,
        'strict': True,
        'schema': {**schema.model_json_schema(), 'additionalProperties': False},
    }
    config = {
        'instructions': instructions,
        'schema': schema.model_json_schema(),
        'reasoning': {'effort': 'none'},
        'max_output_tokens': 1000,
        'store': False,
    }

    def request():
        return client.responses.create(
            model=model, input=evidence, instructions=instructions,
            text={'format': output_format}, reasoning={'effort': 'none'},
            max_output_tokens=1000, store=False,
        )

    return request, model, config

In [ ]:
# Run or replay one OpenAI request, preserving failures and raw response metadata.
def evaluate_openai(client, case, endpoint, *, kind='text', trial=0, warmup=False):
    evidence, identity, query, schema = prepare_openai_input(case, kind)
    typed_questions = to_openai_questions(query)

    if endpoint == 'decisions':
        request, model, config = decisions_request(client, evidence, typed_questions)
        response_schema = None
    elif endpoint == 'responses':
        request, model, config = llm_request(client, evidence, typed_questions, schema)
        response_schema = schema
    else:
        raise ValueError('Unknown endpoint')

    config.update(identity)
    config.update(max_retries=1, timeout=30)
    return run_case(
        request, provider='openai', endpoint=endpoint, model=model,
        example_id=case['id'], config=config, question_names=list(query),
        mode=settings.mode, directory=RUN_DIR, trial=trial, warmup=warmup,
        response_schema=response_schema,
    )

In [ ]:
# Open an API client for live calls, or use saved recordings.
def openai_client():
    return OpenAI(api_key=require_key(settings,'openai'),max_retries=1,timeout=30) if settings.mode=='live' else nullcontext(None)

## 4. Compare one ticket
Change `example_id` to try another ticket. The cell prints the exact state, the answers, and each question’s native probabilities side by side. “Unavailable” means the endpoint did not return that probability, including failed requests. Score distributions use rubric level indices; Noul/predicate shows yes and its complement, no.

In [ ]:
# Ask all four interfaces about the same ticket and compare their native probabilities.
from course_utils.comparison import comparison_tables

example_id = 'dev-mixed'
case = next(ticket for ticket in tickets if ticket['id'] == example_id)
print('State analysed:')
display({'ticket': case['text']})
display({name: question.model_dump(mode='json') for name, question in questions.items()})

text_results = {}
if can_jev:
    with jev_client() as client:
        for model in (settings.jev_old, settings.jev_new):
            text_results[model] = evaluate_ticket(client, case, model)
if can_openai:
    with openai_client() as client:
        text_results['OpenAI Decisions'] = evaluate_openai(client, case, 'decisions')
        text_results['OpenAI LLM'] = evaluate_openai(client, case, 'responses')

if text_results:
    answers, probabilities = comparison_tables(text_results)
    display(answers.fillna('Unavailable'))
    display(probabilities.fillna('Unavailable'))
else:
    print('Configure an API key to run the comparison.')

## 5. Optional: use an image as the evidence
The same comparison works with screenshots. The Jev requests include `TypeSafe-Danger-Zone-Images: true`. OpenAI receives the same image as inline base64. Box localization stays in notebook 03; here we compare category, severity, and error presence.

In [ ]:
# Load screenshots and define image questions with the required opt-in header.
from typesafe_sdk import Image as JevImage, Box
from PIL import Image as PILImage
from course_utils.plots import draw_box
image_cases=[json.loads(line) for line in (ROOT/'data/image_cases.jsonl').read_text().splitlines()]
IMAGE_CATEGORIES={'payment':'A visible payment or checkout error.', 'login':'A visible sign-in or account access error.',
                  'service':'A visible service or connection error/warning.', 'none':'No visible error or warning.'}
SEVERITY=['No error or warning.','Warning, but work can continue.','One specific task is blocked.','The whole service is blocked.']
IMAGE_QUESTIONS={
    'category':Choice(instructions='What kind of error or warning is visible in the screenshot?',criteria=IMAGE_CATEGORIES),
    'severity':Score(instructions='How much work does the visible issue block?',criteria=SEVERITY),
    'error_present':Noul(instructions='Is an error or warning banner visible in the screenshot? Success and empty-state notices are not errors.'),
}
IMAGE_HEADERS={'TypeSafe-Danger-Zone-Images':'true'}

def evaluate_image(client,case,model,*,with_box=False,trial=0,warmup=False):
    path=ROOT/'data'/case['image_path']
    query=dict(IMAGE_QUESTIONS)
    if with_box:
        query['banner']=Box(instructions='Locate the error or warning banner in the screenshot.')
    # The image bytes digest is part of replay identity; labels are never sent to the model.
    config={'image_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
            'state_field':'screenshot','questions':{k:v.model_dump(mode='json') for k,v in query.items()},
            'headers':IMAGE_HEADERS,'max_retries':1,'timeout':30}
    return run_case(lambda:client.system_one(model=model,state={'screenshot':JevImage(path)},
                    questions=query,extra_headers=IMAGE_HEADERS),
        provider='typesafe',endpoint='systemone',model=model,example_id=case['id'],config=config,
        question_names=list(query),mode=settings.mode,directory=RUN_DIR,trial=trial,warmup=warmup)

In [ ]:
# Show one screenshot and compare all four interfaces on matching image questions.
image_case = image_cases[0]
print('State analysed:', image_case['image_path'])
display(PILImage.open(ROOT / 'data' / image_case['image_path']))

image_results = {}
if can_jev:
    with jev_client() as client:
        for model in (settings.jev_old, settings.jev_new):
            image_results[model] = evaluate_image(client, image_case, model)
if can_openai:
    with openai_client() as client:
        image_results['OpenAI Decisions'] = evaluate_openai(client, image_case, 'decisions', kind='image')
        image_results['OpenAI LLM'] = evaluate_openai(client, image_case, 'responses', kind='image')

if image_results:
    image_answers, image_probabilities = comparison_tables(image_results)
    display(image_answers.fillna('Unavailable'))
    display(image_probabilities.fillna('Unavailable'))

## Exercise

Try another ticket. Does the selected team stay the same while its probability changes between Jev versions? Which alternative gains or loses probability? Does OpenAI Decisions agree? Does the LLM select the same answer?

## How to interpret the result

A change in probability is not automatically an improvement. Read the input and rubric. Jev and Decisions scores can be fractional expectations; the LLM selects an integer rubric level. Do not treat its Boolean refund answer as a probability of 0 or 1. One example cannot establish accuracy or calibration.

Choose a decision API when you need typed judgments and distributions; use a generative LLM when you need a written reply. Notebook 02 covers review policies, and notebook 05 combines routing with generation.
